In [0]:
import os 
import re
from pyspark.sql import Row
from pyspark.sql import SparkSession 
import pandas as pd
from pyspark.sql.functions import sum,avg,year,min,max,month,when,col


Bronze_Loacation = '/FileStore/tables/Bronze/Department_salary'
Silver_Location  = 'dbfs:/FileStore/tables/Silver/Department_Salaries'

spark = SparkSession.builder.getOrCreate()


df_Parquet = spark.read.format("Parquet").load(Bronze_Loacation)

try:
    Silver_df = df_Parquet.withColumn("hireyear", year("hireDate")) \
        .withColumn("hireMonth", month("hireDate")) \
        .withColumn(
        "Seniority_Level",
        when(col("Salary") <= 70000, "Junior")
        .when((col("Salary") > 70000) & (col("Salary") < 74000), "Intermediary")
        .when(col("Salary") == 74000, "Senior")
        .otherwise("Executive")  # Add this to handle cases > 74000 or missing
    ) \
    .groupBy("position",
              "department", 
              "hireyear",
               "hireMonth",
               "Salary",
               "Seniority_Level") \
    .agg(
        sum("Salary").alias("Total_Salary"),
        avg("Salary").alias("Avg_Salary"),
        min("Salary").alias("Min_Salary"),
        max("Salary").alias("Max_Salary")
    )




    #display(Silver_df)

#Create a table with the Salaries and Department
    print("Writiting to table")
    Silver_df.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable("Department_Salaries")
    print("Writing to table successful")

    print("Writiting to file")
    Silver_df.write.format('parquet').mode("overwrite").save(Silver_Location)
    print("Writing to file successful")
except Exception as e:
    print("Error:", e)




Writiting to table
Writing to table successful
Writiting to file
Writing to file successful


In [0]:
%sql 

select * from department_salaries

position,department,hireyear,hireMonth,Salary,Seniority_Level,Total_Salary,Avg_Salary,Min_Salary,Max_Salary
Project Manager,Operations,2019,1,70000,Junior,9730000.0,70000.0,70000,70000
Project Manager,Operations,2019,1,71000,Intermediary,9798000.0,71000.0,71000,71000
Project Manager,Operations,2019,1,72000,Intermediary,1.0008E7,72000.0,72000,72000
Project Manager,Operations,2019,1,73000,Intermediary,1.0147E7,73000.0,73000,73000
Project Manager,Operations,2019,1,74000,Senior,1.0286E7,74000.0,74000,74000
Project Manager,Operations,2019,2,70000,Junior,9730000.0,70000.0,70000,70000
Project Manager,Operations,2019,2,71000,Intermediary,9869000.0,71000.0,71000,71000
Project Manager,Operations,2019,2,72000,Intermediary,1.0008E7,72000.0,72000,72000
Project Manager,Operations,2019,2,73000,Intermediary,1.0147E7,73000.0,73000,73000
Project Manager,Operations,2019,2,74000,Senior,1.0286E7,74000.0,74000,74000


In [0]:
display(_sqldf)